# Creating MJO file using LANCZOS filter


### 🔹 STEP 1: Setup


In [ ]:
import xarray as xr
import numpy as np
from scipy.signal import filtfilt

### 🔹 STEP 2: Load anomaly

In [ ]:
file = r"/mnt/hdd4/A_Nilay/RAW_Daily_Prep_1980-2026/anomaly_new.nc"

ds = xr.open_dataset(file, chunks={'time': 100})
anom = ds['tp']

# rename if needed
if 'valid_time' in anom.dims: 
    anom = anom.rename({'valid_time': 'time'})

### 🔹 STEP 3: Create Lanczos weights

In [ ]:
def lanczos_weights(window, cutoff):
    n = (window - 1) // 2
    w = np.zeros(window)

    for i in range(window):
        k = i - n
        if k == 0:
            w[i] = 2 * cutoff
        else:
            w[i] = np.sin(2*np.pi*cutoff*k)/(np.pi*k)
            w[i] *= np.sin(np.pi*k/n)/(np.pi*k/n)
    return w

# parameters
window = 121   # ~4 months
low_cut = 1/100
high_cut = 1/20

w_low = lanczos_weights(window, low_cut)
w_high = lanczos_weights(window, high_cut)

weights = w_high - w_low

### 🔹 STEP 4: Apply filter (IMPORTANT)

In [ ]:
anom = anom.chunk({'time': -1})

def apply_filter(ts, weights):
    return filtfilt(weights, [1.0], ts, axis=0, padtype='odd')

mjo = xr.apply_ufunc(
    apply_filter,
    anom,
    input_core_dims=[['time']],
    output_core_dims=[['time']],
    vectorize=True,
    dask="parallelized",
    kwargs={'weights': weights},
    output_dtypes=[float],
    dask_gufunc_kwargs={'allow_rechunk': True}
)

In [ ]:
print("Anomaly max:", float(anom.max().compute()))
print("MJO max:", float(mjo.max().compute()))

### 🔹 STEP 5: Creating only MJO file

In [ ]:
import xarray as xr

# ==============================
# ENSURE CORRECT DIMENSION ORDER
# ==============================
mjo = mjo.transpose('time', 'latitude', 'longitude')

# ==============================
# SET VARIABLE NAME + ATTRIBUTES
# ==============================
mjo.name = "tp_mjo"
mjo.attrs['long_name'] = "MJO filtered precipitation (20-100 day)"
mjo.attrs['units'] = "m"

# ==============================
# CONVERT TO DATASET (IMPORTANT)
# ==============================
ds_out = mjo.to_dataset()

# ==============================
# SAVE (OPTIMIZED)
# ==============================
encoding = {
    "tp_mjo": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32"
    }
}

ds_out.to_netcdf("lanczos_mjo_only.nc", encoding=encoding)

print("✅ MJO file created: lanczos_mjo_only.nc")

### 🔹 STEP 6: Detecting any date gaps in the MJO file

In [ ]:
import xarray as xr
import pandas as pd

# ==============================
# FILE PATH
# ==============================
file = r"/mnt/hdd4/A_Nilay/RAW_Daily_Prep_1980-2026/lanczos_mjo_only.nc"

# ==============================
# LOAD DATA
# ==============================
ds = xr.open_dataset(file)

time = pd.to_datetime(ds['time'].values)

# ==============================
# LIMIT TO 2025-12-31
# ==============================
end_date = pd.Timestamp("2025-12-31")
time = time[time <= end_date]

# ==============================
# CREATE IDEAL CONTINUOUS SERIES
# ==============================
expected_time = pd.date_range(
    start=time.min(),
    end=end_date,
    freq='D'
)

# ==============================
# FIND MISSING DATES
# ==============================
missing_dates = expected_time.difference(time)

# ==============================
# OUTPUT
# ==============================
print("\n===== GAP CHECK (UPTO 2025-12-31) =====")

if len(missing_dates) == 0:
    print("✅ No missing dates — data is continuous")
else:
    print(f"❌ Missing {len(missing_dates)} dates:\n")
    print(missing_dates)

# ==============================
# EXTRA: CHECK DUPLICATES
# ==============================
duplicates = pd.Series(time).duplicated().sum()

print("\n===== DUPLICATE CHECK =====")
print("Duplicate timestamps:", duplicates)

### 🔹 STEP 7: Creating non MJO file: (non MJO = anomay - lanczos MJO)

In [ ]:
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

# ==============================
# USE THREAD SCHEDULER
# ==============================
dask.config.set(scheduler='threads')

# ==============================
# FILE PATHS
# ==============================
anom_file = "/mnt/hdd4/A_Nilay/RAW_Daily_Prep_1980-2026/anomaly_new.nc"
mjo_file  = "/mnt/hdd4/A_Nilay/RAW_Daily_Prep_1980-2026/lanczos_mjo_only.nc"

# ==============================
# LOAD DATA
# ==============================
anom = xr.open_dataset(anom_file)['tp']
mjo  = xr.open_dataset(mjo_file)['tp_mjo']

# ==============================
# FIX DIMENSIONS
# ==============================
if 'valid_time' in anom.dims:
    anom = anom.rename({'valid_time': 'time'})

# ==============================
# RECHUNK (OPTIMAL)
# ==============================
anom = anom.chunk({'time': 2000})
mjo  = mjo.chunk({'time': 2000})

# ==============================
# ALIGN
# ==============================
anom, mjo = xr.align(anom, mjo)

print("✅ Alignment done")

# ==============================
# COMPUTE NON-MJO
# ==============================
non_mjo = anom - mjo

non_mjo.name = "tp_non_mjo"
non_mjo.attrs['units'] = "m"

# ==============================
# SAVE WITH PROGRESS BAR
# ==============================
output_file = "/mnt/hdd4/A_Nilay/RAW_Daily_Prep_1980-2026/non_mjo.nc"

with ProgressBar():
    non_mjo.to_netcdf(
        output_file,
        engine="netcdf4",
        encoding={
            "tp_non_mjo": {
                "zlib": False,
                "dtype": "float32"
            }
        }
    )

print("✅ DONE: non_mjo.nc created")

# ==============================
# VALIDATION (OPTIONAL)
# ==============================
with ProgressBar():
    error = (anom - (mjo + non_mjo)).mean().compute()

print("Residual error:", float(error.values))

# Attempting the validation process of newly created non mjo file =>

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

# ==============================
# 📂 FILE PATHS
# ==============================
FOLDER = "/mnt/hdd4/A_Nilay/RAW_Daily_Prep_1980-2026/"

f_raw = FOLDER + "ERA5_merged_1980_2026_new.nc"
f_mjo = FOLDER + "lanczos_mjo_only.nc"
f_non = FOLDER + "non_mjo.nc"

# ==============================
# 📍 USER INPUT
# ==============================
TARGET_LAT = 22.3
TARGET_LON = 87.3

START = "1980-10-01"
END   = "2025-10-31"

# ==============================
# 📖 LOAD DATA
# ==============================
ds_raw = xr.open_dataset(f_raw)
ds_mjo = xr.open_dataset(f_mjo)
ds_non = xr.open_dataset(f_non)

raw = ds_raw['tp']
mjo = ds_mjo['tp_mjo']
non = ds_non['tp_non_mjo']

# ==============================
# 🧠 FIX TIME DIM
# ==============================
if 'valid_time' in raw.dims:
    raw = raw.rename({'valid_time': 'time'})

# ==============================
# 📍 SELECT GRID
# ==============================
raw_pt = raw.sel(latitude=TARGET_LAT, longitude=TARGET_LON, method='nearest')
mjo_pt = mjo.sel(latitude=TARGET_LAT, longitude=TARGET_LON, method='nearest')
non_pt = non.sel(latitude=TARGET_LAT, longitude=TARGET_LON, method='nearest')

# ==============================
# 🕒 TIME FILTER
# ==============================
raw_pt = raw_pt.sel(time=slice(START, END))
mjo_pt = mjo_pt.sel(time=slice(START, END))
non_pt = non_pt.sel(time=slice(START, END))

# ==============================
# 📊 CONVERT TO mm/day (ONLY FOR PLOT)
# ==============================
raw_mm = raw_pt * 1000
mjo_mm = mjo_pt * 1000
non_mm = non_pt * 1000

# ==============================
# 📈 PLOT
# ==============================
plt.figure(figsize=(14,6))

# Plot order matters (overlay clarity)
plt.plot(raw_mm['time'], raw_mm, label='Raw', linewidth=2, color='blue')
plt.plot(non_mm['time'], non_mm, label='Non-MJO', linewidth=2, color='green')
plt.plot(mjo_mm['time'], mjo_mm, label='MJO', linewidth=2, color='red')

plt.axhline(0, color='black', linewidth=0.5)

plt.title(f"Raw vs MJO vs Non-MJO at ({TARGET_LAT}, {TARGET_LON})")
plt.ylabel("mm/day")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

# ==============================
# CLOSE
# ==============================
ds_raw.close()
ds_mjo.close()
ds_non.close()